# Thesis figures

Plotting code of the thesis chapters in figure order, run on `data/plot_sources/`, saved to `figures/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

FIG = Path("../figures")
FIG.mkdir(exist_ok=True)


def save(name):
    plt.savefig(FIG / f"{name}.png", dpi=200, bbox_inches="tight")
    plt.close("all")

## Chapter 3

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

path = Path("../data/plot_sources/binary_replication.csv")

df = pd.read_csv(path).sort_values(["run", "step"])

trajectories = df.pivot(
    index="step",
    columns="run",
    values="coordination_level",
)

mean_trajectory = trajectories.mean(axis=1)
distance_to_mean = (
    trajectories.sub(mean_trajectory, axis=0) ** 2
).mean(axis=0)

representative_run = distance_to_mean.idxmin()

representative = df[
    df["run"] == representative_run
].sort_values("step")

final_values = (
    df.groupby("run", as_index=False)
    .tail(1)["coordination_level"]
)

style = {
    "font.family": "serif",
    "font.serif": ["STIX Two Text", "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.linewidth": 0.7,
}

with plt.rc_context(style):
    fig = plt.figure(figsize=(6.6, 3.6))
    grid = fig.add_gridspec(
        1,
        2,
        width_ratios=[5, 1],
        wspace=0.04,
    )

    ax = fig.add_subplot(grid[0, 0])
    ax_box = fig.add_subplot(grid[0, 1], sharey=ax)

    ax.plot(
        representative["time"],
        representative["coordination_level"],
        color="#1976A3",
        linewidth=1.4,
    )

    ax.axhline(
        1,
        color="#777777",
        linestyle=(0, (3, 2)),
        linewidth=0.8,
    )

    ax_box.boxplot(
        final_values,
        vert=True,
        widths=0.48,
        patch_artist=True,
        boxprops={
            "facecolor": "#B9D7E8",
            "edgecolor": "#7699AD",
            "linewidth": 0.8,
        },
        medianprops={
            "color": "#1976A3",
            "linewidth": 2.2,
        },
        whiskerprops={
            "color": "#777777",
            "linewidth": 0.8,
        },
        capprops={
            "color": "#777777",
            "linewidth": 0.8,
        },
        flierprops={
            "marker": "o",
            "markerfacecolor": "#1976A3",
            "markeredgecolor": "none",
            "markersize": 3,
        },
    )

    ax.set_xlim(0, 10)
    ax.set_ylim(-0.02, 1.04)
    ax.set_xticks(range(0, 11, 2))
    ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])

    ax.set_xlabel(r"Time $t$")
    ax.set_ylabel(r"Coordination level $C(t)=|m(t)|$")

    ax.grid(
        axis="y",
        color="#D7D7D7",
        linewidth=0.5,
        alpha=0.75,
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax_box.set_xticks([])
    ax_box.set_xlabel("Final\ncoordination", fontsize=8)
    ax_box.tick_params(axis="y", left=False, labelleft=False)
    ax_box.spines["top"].set_visible(False)
    ax_box.spines["right"].set_visible(False)

    fig.subplots_adjust(
        left=0.11,
        right=0.98,
        bottom=0.18,
        top=0.98,
    )

    save("fig-llama70b-replication")

In [ ]:
import json

import numpy as np

counts_path = Path("../data/plot_sources/binary_anchor_counts.csv")
fits_path = Path("../data/plot_sources/neutral_parameters.csv")

# the public table has one row per (state, reply) with a count n;
# expand it back to one row per query so the code below stays as it was
anchor_counts = pd.read_csv(counts_path)
neutral = anchor_counts.loc[anchor_counts.index.repeat(anchor_counts["n"])]
neutral = neutral.drop(columns="n").reset_index(drop=True)
neutral = neutral[neutral["valid"]].copy()
fits = pd.read_csv(fits_path)

binary = neutral[(neutral["q"] == 2) & (neutral["n_display"] == 49)].copy()
binary["shares_options_list"] = binary["shares_options"].apply(json.loads)
binary["m"] = binary["shares_options_list"].apply(
    lambda x: round(x[0] - x[1], 10)
)
binary["choose_first"] = binary["chosen_display"].eq("aa")
binary_response = (
    binary.groupby(["model_label", "m"], as_index=False)
    .agg(p_first=("choose_first", "mean"), n=("choose_first", "size"))
)

z = 1.96
denom = 1 + z**2 / binary_response["n"]
center = (
    binary_response["p_first"] + z**2 / (2 * binary_response["n"])
) / denom
half = z * np.sqrt(
    binary_response["p_first"]
    * (1 - binary_response["p_first"])
    / binary_response["n"]
    + z**2 / (4 * binary_response["n"] ** 2)
) / denom
binary_response["ci_low"] = center - half
binary_response["ci_high"] = center + half

anchor_models = ["gemma4_31B_dense", "qwen25_7b_it", "llama3_70b_awq"]
anchor_model_names = {
    "gemma4_31B_dense": "Gemma 4 31B Dense",
    "qwen25_7b_it": "Qwen 2.5 7B",
    "llama3_70b_awq": "Llama 3 70B",
}
m_grid = np.linspace(-1, 1, 300)

with plt.rc_context(style):
    fig_a, axes_a = plt.subplots(
        1,
        3,
        figsize=(12.5, 4),
        sharex=True,
        sharey=True,
    )

    for ax, model in zip(axes_a, anchor_models):
        g = binary_response[
            binary_response["model_label"] == model
        ].sort_values("m")
        beta_potts = fits[
            (fits["model_label"] == model)
            & (fits["q"] == 2)
            & (fits["n_display"] == 49)
        ]["beta_m1"].iloc[0]
        # fits from before the convention change lack the beta_c column and store twice the thesis value
        beta_binary = beta_potts / 2 if "beta_c" not in fits.columns else beta_potts

        ax.plot(g["m"], g["p_first"], "o", ms=5, color="#1F77B4")
        # ax.errorbar(
        #     g["m"],
        #     g["p_first"],
        #     yerr=np.vstack(
        #         [
        #             g["p_first"] - g["ci_low"],
        #             g["ci_high"] - g["p_first"],
        #         ]
        #     ),
        #     fmt="o",
        #     ms=5,
        #     capsize=2,
        #     color="#1F77B4",
        # )
        ax.plot(
            m_grid,
            0.5 * (1 + np.tanh(beta_binary * m_grid)),
            color="#D62728",
            linewidth=2,
        )
        ax.set_title(
            f"{anchor_model_names[model]}\n"
            f"$\\hat{{\\beta}}$ = {beta_binary:.2f}"
        )
        ax.set_xlabel("Collective opinion m")
        ax.set_ylabel(r"Adoption probability $P(m)$")
        ax.tick_params(axis="y", labelleft=True)
        ax.set_xlim(-1.05, 1.05)
        ax.set_ylim(-0.03, 1.03)
        ax.grid(alpha=0.18)

    fig_a.suptitle("Binary Validation, q=2", fontsize=15)
    fig_a.tight_layout(rect=[0, 0, 1, 0.92])
    save("fig-binary-anchor")

## Chapter 4

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import ScalarFormatter

plot_data = Path("../data/plot_sources")
dyn_data = Path("../data/plot_sources")

THESIS_STYLE = {
    "font.family": "serif",
    "font.serif": ["STIX Two Text", "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 9,
    "axes.linewidth": 0.7,
}

def style_axis(ax):
    ax.grid(alpha=0.18)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

q_values = [3, 10, 50]

models = [
    "llama31_8b_it",
    "llama3_70b_awq",
    "qwen25_7b_it",
    "qwen25_32b_it",
    "gemma4_E4B_it",
    "gemma4_31B_dense",
]

model_names = {
    "llama31_8b_it": "Llama 3.1 8B",
    "llama3_70b_awq": "Llama 3 70B",
    "qwen25_7b_it": "Qwen 2.5 7B",
    "qwen25_32b_it": "Qwen 2.5 32B",
    "gemma4_E4B_it": "Gemma 4 E4B",
    "gemma4_31B_dense": "Gemma 4 31B Dense",
}

model_colors = {
    "llama31_8b_it": "#8C564B",
    "llama3_70b_awq": "#2CA02C",
    "qwen25_7b_it": "#FF7F0E",
    "qwen25_32b_it": "#D62728",
    "gemma4_E4B_it": "#17BECF",
    "gemma4_31B_dense": "#1F77B4",
}

# one color per q
q_colors = {2: "#6A3D9A", 3: "#33A02C", 10: "#1F78B4", 50: "#E31A1C", 100: "#FF7F00"}

trajectory_data = pd.read_csv(dyn_data / "dynamics_trajectories.csv")
summary = pd.read_csv(dyn_data / "dynamics_summary.csv")
data = trajectory_data[
    trajectory_data["record_type"] == "regular"
].copy()

# representative group size 
N_show = 100

In [ ]:
q_rows = [3, 10, 50]
N_over = 100

def order_param(s, q): return (q*s-1)/(q-1)

with plt.rc_context(THESIS_STYLE):
    fig, axes = plt.subplots(len(q_rows), 1, figsize=(9.0, 6.0), sharex=True)
    for ax, q in zip(axes, q_rows):
        panel = data[(data["q"]==q)&(data["N"]==N_over)]
        for model in models:
            g = panel[panel["model_label"]==model]
            if g.empty: continue
            fin = g.sort_values("time").groupby("run")["leader_share"].last()
            rep = (fin - fin.median()).abs().idxmin()
            t = g[g["run"]==rep].sort_values("time")
            m = order_param(t["leader_share"], q)
            done = t[np.isclose(t["leader_share"],1)]
            if not done.empty:
                t1 = done["time"].iloc[0]
                keep = t["time"]<=t1
                ax.plot(t.loc[keep,"time"], m[keep], color=model_colors[model], lw=1.4)
                ax.scatter(t1, 1, marker="v", s=22, color=model_colors[model], zorder=4)
            else:
                ax.plot(t["time"], m, color=model_colors[model], lw=1.4)
        ax.axhline(1, color="0.4", ls="--", lw=0.8)
        ax.set_xscale("function", functions=(np.log1p, np.expm1))
        ax.set_xlim(0,100); ax.set_xticks([0,1,2,5,10,20,50,100]); ax.set_xticklabels([0,1,2,5,10,20,50,100])
        ax.set_ylim(-0.04,1.06)
        ax.set_xlabel(r"Time $t$")
        ax.set_ylabel(r"$C(t)$")
        ax.set_title(f"$q={q}$", fontsize=9, loc="left")
        ax.tick_params(axis="x", labelbottom=True)
        ax.grid(alpha=0.18); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    handles=[Line2D([0],[0],color=model_colors[m],lw=1.6,label=model_names[m]) for m in models]
    fig.legend(handles=handles, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.0))
    fig.tight_layout(rect=[0,0,1,0.955])
    save("fig-results-coordination-overview")

In [ ]:
with plt.rc_context(THESIS_STYLE):
    fig_s, axes_s = plt.subplots(2, 3, figsize=(10.5, 5.8), sharex=True, sharey=True)

    for ax, model in zip(axes_s.flat, models):
        panel = data[(data["model_label"] == model) & (data["N"] == N_show)]
        for q in q_values:
            trajectories = panel[panel["q"] == q].sort_values("time")
            if trajectories.empty:
                continue
            trajectory = (
                trajectories.groupby("time", as_index=False)["leader_share"]
                .mean().sort_values("time")
            )
            coordinated = trajectory[np.isclose(trajectory["leader_share"], 1)]
            if not coordinated.empty:
                first_time = coordinated.iloc[0]["time"]
                trajectory = trajectory[trajectory["time"] <= first_time]
            ax.plot(
                trajectory["time"], trajectory["leader_share"],
                color=q_colors[q], linewidth=1.6,
            )
            if not coordinated.empty:
                ax.scatter(first_time, 1, marker="v", s=26, color=q_colors[q], zorder=4)

        ax.axhline(1, color="0.4", linestyle="--", linewidth=0.8)
        ax.set_xscale("function", functions=(np.log1p, np.expm1))
        ax.set_xlim(0, 100)
        ax.set_xticks([0, 1, 2, 5, 10, 20, 50, 100])
        ax.set_xticklabels([0, 1, 2, 5, 10, 20, 50, 100])
        ax.set_ylim(0, 1.05)
        ax.set_yticks([0, 0.25, 0.5, 0.75, 1])
        ax.set_xlabel(r"Time $t$")
        ax.set_title(model_names[model])
        ax.tick_params(axis="x", labelbottom=True)
        style_axis(ax)

    for ax in axes_s[:, 0]:
        ax.set_ylabel(r"Leader share $s(t)$")

    q_legend = [
        Line2D([0], [0], color=q_colors[q], linewidth=1.8, label=f"$q={q}$")
        for q in q_values
    ]
    fig_s.legend(
        handles=q_legend, loc="upper center", ncol=3,
        frameon=False, bbox_to_anchor=(0.5, 1.02),
    )
    fig_s.tight_layout(rect=[0, 0, 1, 0.95])
    save("fig-results-leader-share")

In [ ]:
with plt.rc_context(THESIS_STYLE):
    fig_ey, axes_ey = plt.subplots(1, 3, figsize=(10.5, 3.5), sharex=True)

    for ax, q in zip(axes_ey, q_values):
        panel = data[(data["q"] == q) & (data["N"] == N_show)]
        for model in models:
            trajectories = panel[panel["model_label"] == model].sort_values("time")
            if trajectories.empty:
                continue
            trajectory = (
                trajectories.groupby("time", as_index=False)["n_active_opinions"]
                .mean().sort_values("time")
            )
            coordinated = trajectory[trajectory["n_active_opinions"] == 1]
            if not coordinated.empty:
                first_time = coordinated.iloc[0]["time"]
                trajectory = trajectory[trajectory["time"] <= first_time]
            ax.step(
                trajectory["time"], trajectory["n_active_opinions"],
                where="post", color=model_colors[model], linewidth=1.5,
            )
            if not coordinated.empty:
                ax.scatter(
                    first_time, 1, marker="v", s=24,
                    color=model_colors[model], zorder=4,
                )

        ax.axhline(1, color="0.4", linestyle="--", linewidth=0.8)
        ax.set_xscale("function", functions=(np.log1p, np.expm1))
        ax.set_yscale("log")
        ax.set_xlim(0, 100)
        ax.set_xticks([0, 1, 2, 5, 10, 20, 50, 100])
        ax.set_xticklabels([0, 1, 2, 5, 10, 20, 50, 100])
        y_ticks = {3: [1, 2, 3], 10: [1, 2, 5, 10], 50: [1, 2, 5, 10, 20, 50]}[q]
        ax.set_ylim(0.9, q * 1.15)
        ax.set_yticks(y_ticks)
        ax.set_yticklabels(y_ticks)
        ax.set_title(f"$q={q}$")
        ax.set_xlabel(r"Time $t$")
        style_axis(ax)

    axes_ey[0].set_ylabel(r"Active opinions $K(t)$")
    model_legend = [
        Line2D([0], [0], color=model_colors[m], linewidth=1.6, label=model_names[m])
        for m in models
    ]
    fig_ey.legend(
        handles=model_legend, loc="upper center", ncol=3,
        frameon=False, bbox_to_anchor=(0.5, 1.14),
    )
    fig_ey.tight_layout(rect=[0, 0, 1, 0.99])
    save("fig-results-opinion-extinction")

In [ ]:
per_run = summary[[
    "model_label", "q", "N", "run",
    "event_observed", "consensus_time_sweeps", "scheduled_horizon_sweeps",
]].copy()
per_run["plot_time"] = per_run["consensus_time_sweeps"].where(
    per_run["event_observed"], per_run["scheduled_horizon_sweeps"]
)
run_times = per_run.groupby(["model_label", "q", "N"], as_index=False).agg(
    frac_coordinated=("event_observed", "mean"),
    horizon=("scheduled_horizon_sweeps", "max"),
)
med_finishers = (
    per_run[per_run["event_observed"]]
    .groupby(["model_label", "q", "N"], as_index=False)
    .agg(consensus_time=("consensus_time_sweeps", "median"))
)
run_times = run_times.merge(med_finishers, on=["model_label", "q", "N"], how="left")
run_times["coordinated"] = run_times["frac_coordinated"] > 0
run_times["plot_time"] = run_times["consensus_time"].fillna(run_times["horizon"])
horizon = run_times["horizon"].max()
censor_offsets = {3: 0.96, 10: 1.00, 50: 1.04}


# the four models with observable consensus times; the Llamas never
T_models = ["qwen25_7b_it", "qwen25_32b_it", "gemma4_E4B_it", "gemma4_31B_dense"]

with plt.rc_context(THESIS_STYLE):
    fig_t, axes_t = plt.subplots(2, 2, figsize=(8.4, 6.4), sharex=True, sharey=True)

    for ax, model in zip(axes_t.flat, T_models):
        for q in q_values:
            group = run_times[
                (run_times["model_label"] == model) & (run_times["q"] == q)
            ].sort_values("N")
            if group.empty:
                continue
            reached = group["coordinated"]
            ax.plot(
                group.loc[reached, "N"], group.loc[reached, "consensus_time"],
                color=q_colors[q], marker="o", linewidth=1.5, markersize=4,
            )
            ax.scatter(
                group.loc[~reached, "N"] * censor_offsets[q],
                group.loc[~reached, "plot_time"], marker="^",
                facecolors="white", edgecolors=q_colors[q],
                linewidths=1.3, s=38, zorder=4,
            )

        ax.axhline(horizon, color="0.55", linestyle="--", linewidth=0.8)
        ax.set_title(model_names[model])
        ax.set_xscale("log", base=2)
        ax.set_yscale("log")
        ax.set_xlim(20, 1000)
        ax.set_xticks([25, 50, 100, 200, 400, 800])
        ax.set_ylim(2, 130)
        ax.set_yticks([2, 5, 10, 20, 50, 100])
        ax.set_xlabel(r"Group size $N$")
        ax.tick_params(axis="x", labelbottom=True)
        ax.xaxis.set_major_formatter(ScalarFormatter())
        ax.yaxis.set_major_formatter(ScalarFormatter())
        style_axis(ax)

    for ax in axes_t[:, 0]:
        ax.set_ylabel(r"$T(N)$")

    time_legend = [
        Line2D(
            [0], [0], color=q_colors[q], marker="o",
            linewidth=1.5, markersize=4, label=f"$q={q}$",
        )
        for q in q_values
    ]
    time_legend.append(
        Line2D(
            [0], [0], marker="^", markerfacecolor="white",
            markeredgecolor="0.3", linestyle="none",
            label=rf"$T>{int(horizon)}$",
        )
    )
    fig_t.legend(
        handles=time_legend, loc="upper center", ncol=4,
        frameon=False, bbox_to_anchor=(0.5, 1.02),
    )
    fig_t.tight_layout(rect=[0, 0, 1, 0.96])
    save("fig-results-consensus-time")

In [ ]:

# expand it back to one row per query so the code below stays as it was
response_counts = pd.read_csv(plot_data / "neutral_response_counts.csv")
queries = response_counts.loc[response_counts.index.repeat(response_counts["n"])]
queries = queries.drop(columns="n").reset_index(drop=True)
queries["mode"] = "neutral"
queries["valid"] = queries["chosen_role"].notna()
fits = pd.read_csv(plot_data / "neutral_parameters.csv")

legacy_beta_units = "beta_c" not in fits.columns
if legacy_beta_units:
    for column in [
        "beta_m1", "beta_ci_lo", "beta_ci_hi", "beta_m2", "beta_m3",
    ]:
        if column in fits.columns:
            fits[column] = fits[column] / 2.0
    fits["beta_c"] = fits["q"].map(
        lambda q: 1.0 if int(q) == 2
        else (q - 1) / (q - 2) * np.log(q - 1)
    )

neutral = queries[
    (queries["mode"] == "neutral") & (queries["valid"] == True)
].copy()
neutral["is_leader"] = neutral["chosen_role"].eq(0)

response = (
    neutral.groupby(
        ["model_label", "q", "n_display", "rel"], as_index=False,
    )
    .agg(
        p_leader=("is_leader", "mean"),
        n=("is_leader", "size"),
        leader_share=("leader_share", "first"),
    )
    .merge(
        fits[["model_label", "q", "n_display", "beta_m1"]],
        on=["model_label", "q", "n_display"],
    )
)

def potts_leader_curve(beta, q, leader_share):
    rest_share = (1 - leader_share) / (q - 1)
    return 1 / (
        1 + (q - 1) * np.exp(-2.0 * beta * (leader_share - rest_share))
    )

# q values shown per group size in the beta(q) figure
response_blocks = {49: [2, 3, 5, 10, 25], 199: [2, 3, 5, 10, 25, 50, 100], 799: [2, 3, 5, 10, 25, 50, 100]}

In [ ]:
response_show_q = [2, 3, 10, 50]
response_display = 199

with plt.rc_context(THESIS_STYLE):
    fig_r, axes_r = plt.subplots(2, 3, figsize=(10.5, 6.0), sharex=True, sharey=True)

    for ax, model in zip(axes_r.flat, models):
        for q in response_show_q:
            group = response[
                (response["model_label"] == model)
                & (response["q"] == q)
                & (response["n_display"] == response_display)
            ].sort_values("leader_share")
            if group.empty:
                continue
            ax.plot(
                group["leader_share"], group["p_leader"], "o",
                ms=3.4, color=q_colors[q],
            )
            beta = group["beta_m1"].iloc[0]
            s_grid = np.linspace(
                group["leader_share"].min(), group["leader_share"].max(), 250,
            )
            ax.plot(
                s_grid, potts_leader_curve(beta, q, s_grid),
                color=q_colors[q], linewidth=1.5,
            )

        ax.set_title(model_names[model])
        ax.set_xlim(0, 1.02)
        ax.set_ylim(-0.02, 1.03)
        ax.set_xlabel(r"Leader share $s$")
        ax.tick_params(axis="x", labelbottom=True)
        style_axis(ax)

    for ax in axes_r[:, 0]:
        ax.set_ylabel(r"$P_{\mathrm{lead}}(s)$")

    response_legend = [
        Line2D(
            [0], [0], color=q_colors[q], linewidth=1.6,
            marker="o", markersize=3.5, label=f"$q={q}$",
        )
        for q in response_show_q
    ]
    fig_r.legend(
        handles=response_legend, loc="upper center", ncol=4,
        frameon=False, bbox_to_anchor=(0.5, 1.02),
    )
    fig_r.tight_layout(rect=[0, 0, 1, 0.96])
    save("fig-results-response-n200")

In [ ]:
with plt.rc_context(THESIS_STYLE):
    fig_b, axes_b = plt.subplots(1, 3, figsize=(10.5, 3.8), sharey=True)

    for ax, (n_display, qs) in zip(axes_b, response_blocks.items()):
        panel = fits[(fits["n_display"] == n_display) & (fits["q"].isin(qs))]
        for model in models:
            group = panel[panel["model_label"] == model].sort_values("q")
            group = group[group["beta_m1"] >= 0.1]
            if group.empty:
                continue
            ax.plot(
                group["q"], group["beta_m1"], marker="o", linewidth=1.4,
                markersize=4, color=model_colors[model], label=model_names[model],
            )
        threshold_q = np.array(sorted(qs), dtype=float)
        threshold_beta = np.array([
            1.0 if q == 2 else (q - 1) / (q - 2) * np.log(q - 1)
            for q in threshold_q
        ])
        ax.plot(
            threshold_q, threshold_beta, color="black", linestyle="--",
            linewidth=1.1, label=r"$\beta_c(q)$",
        )
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xticks(qs)
        ax.set_xticklabels(qs)
        ax.xaxis.set_minor_locator(plt.NullLocator())
        ax.set_yticks([0.5, 1, 2, 5, 10, 20, 50, 100])
        ax.yaxis.set_major_formatter(ScalarFormatter())
        ax.set_xlabel(r"Number of opinions $q$")
        ax.set_title(f"$N={n_display + 1}$")
        style_axis(ax)

    axes_b[0].set_ylabel(r"Fitted $\hat\beta$")
    axes_b[0].set_ylim(0.3, 180)

    handles, labels = axes_b[0].get_legend_handles_labels()
    fig_b.legend(
        handles, labels, loc="upper center", ncol=4,
        frameon=False, bbox_to_anchor=(0.5, 1.12),
    )
    fig_b.tight_layout(rect=[0, 0, 1, 0.99])
    save("fig-results-beta-q")

In [ ]:
binary = fits[fits["q"] == 2].copy()
binary["N"] = binary["n_display"] + 1

with plt.rc_context(THESIS_STYLE):
    fig_p, ax_p = plt.subplots(figsize=(6.8, 4.4))

    N_grid = np.linspace(20, 1000, 300)
    beta_t = 0.5 * np.log(N_grid)
    y_top = 8.5  # fixed cap
    y_bot = -0.5

    ax_p.fill_between(N_grid, beta_t, y_top, color="#DDF0DC", zorder=0)
    ax_p.fill_between(N_grid, 1.0, beta_t, color="#FBF3D5", zorder=0)
    ax_p.fill_between(N_grid, y_bot, 1.0, color="#DCE9F5", zorder=0)
    ax_p.plot(N_grid, beta_t, color="0.45", linestyle="--", linewidth=1.0)
    ax_p.axhline(1.0, color="0.45", linestyle="--", linewidth=1.0)

    for model in models:
        group = binary[binary["model_label"] == model].sort_values("N")
        if group.empty:
            continue
        interior = group[
            (group["beta_is_lower_bound"] != True) & (group["beta_m1"] <= y_top)
        ]
        clipped = group[
            (group["beta_is_lower_bound"] == True) | (group["beta_m1"] > y_top)
        ]
        yerr = np.vstack([
            interior["beta_m1"] - interior["beta_ci_lo"],
            np.minimum(interior["beta_ci_hi"], y_top) - interior["beta_m1"],
        ])
        ax_p.errorbar(
            interior["N"], interior["beta_m1"], yerr=yerr,
            color=model_colors[model], marker="o", markersize=4,
            linewidth=1.5, capsize=2.5, label=model_names[model], zorder=3,
        )
        for _, row in clipped.iterrows():
            y_val = min(row["beta_m1"], y_top * 0.96)
            ax_p.scatter(
                row["N"], y_val, marker="^", s=48, facecolors="white",
                edgecolors=model_colors[model], linewidths=1.5, zorder=5,
            )
            ax_p.annotate(
                rf"$\geq{row['beta_ci_lo']:.0f}$", xy=(row["N"], y_val),
                xytext=(0, 7), textcoords="offset points",
                ha="center", fontsize=7.5, color=model_colors[model],
            )

    ax_p.text(0.76, 0.70, "Coordinated", transform=ax_p.transAxes,
              fontsize=9, color="#3A6B38")
    ax_p.text(0.60, 0.355, "Partially coordinated", transform=ax_p.transAxes,
              fontsize=9, color="#8A6D1D")
    ax_p.text(0.62, 0.10, "Uncoordinated", transform=ax_p.transAxes,
              fontsize=9, color="#39597A")
    ax_p.annotate(r"$\beta_t(N)$", xy=(0.03, 0.51),
                  xycoords="axes fraction", fontsize=8.5, color="0.35")
    ax_p.annotate(r"$\beta_c$", xy=(0.03, 0.245),
                  xycoords="axes fraction", fontsize=8.5, color="0.35")

    ax_p.set_xscale("log")
    ax_p.set_xlim(20, 1000)
    ax_p.set_xticks([25, 50, 100, 200, 400, 800])
    ax_p.xaxis.set_major_formatter(ScalarFormatter())
    ax_p.xaxis.set_minor_locator(plt.NullLocator())
    ax_p.set_ylim(y_bot, y_top)
    ax_p.set_xlabel(r"Group size $N$")
    ax_p.set_ylabel(r"Conformity strength $\hat\beta$ at $q=2$")
    ax_p.grid(alpha=0.15)
    ax_p.spines["top"].set_visible(False)
    ax_p.spines["right"].set_visible(False)
    handles_p, labels_p = ax_p.get_legend_handles_labels()
    handles_p.append(Line2D([0], [0], marker="^", markerfacecolor="white", markeredgecolor="0.3",
                            linestyle="none", markersize=6.5, label="poorly determined estimate, above axis range"))
    labels_p.append("poorly determined estimate, above axis range")
    fig_p.legend(handles_p, labels_p, loc="upper center", frameon=False, ncol=4,
                 fontsize=8, bbox_to_anchor=(0.5, 1.06))
    fig_p.tight_layout()
    save("fig-results-binary-phase")

In [ ]:
fits_N = fits.copy()
fits_N["N"] = fits_N["n_display"] + 1

def beta_c(q): return 1.0 if q==2 else (q-1)/(q-2)*np.log(q-1)

with plt.rc_context(THESIS_STYLE):
    fig_mq, axes_mq = plt.subplots(1, 3, figsize=(10.5, 3.9), sharex=True, sharey=True)
    for ax, q in zip(axes_mq, [3, 10, 50]):
        N_grid = np.linspace(20, 1000, 300)
        bt = 0.5*np.log((q-1)*N_grid)
        bc = beta_c(q)
        y_bot, y_top = 0.15, 200
        ax.fill_between(N_grid, bt, y_top, color="#DDF0DC", zorder=0)
        ax.fill_between(N_grid, bc, bt, color="#FBF3D5", zorder=0)
        ax.fill_between(N_grid, y_bot, bc, color="#DCE9F5", zorder=0)
        ax.plot(N_grid, bt, color="0.45", ls="--", lw=1.0)
        ax.axhline(bc, color="0.45", ls="--", lw=1.0)
        for model in models:
            g = fits_N[(fits_N.model_label==model)&(fits_N.q==q)].sort_values("N")
            g = g[g.beta_m1 >= 0.15]
            if g.empty: continue
            interior = g[g.beta_is_lower_bound != True]
            bounded = g[g.beta_is_lower_bound == True]
            yerr = np.vstack([interior.beta_m1-interior.beta_ci_lo, interior.beta_ci_hi-interior.beta_m1])
            ax.errorbar(interior.N, interior.beta_m1, yerr=yerr, color=model_colors[model],
                        marker="o", ms=3.6, lw=1.4, capsize=2, zorder=3, label=model_names[model])
            if not bounded.empty:
                ax.scatter(bounded.N, bounded.beta_m1, marker="^", s=42, facecolors="white",
                           edgecolors=model_colors[model], linewidths=1.4, zorder=5)
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlim(20, 1000); ax.set_ylim(y_bot, y_top)
        ax.set_xticks([25,50,100,200,400,800]); ax.xaxis.set_major_formatter(ScalarFormatter())
        ax.xaxis.set_minor_locator(plt.NullLocator())
        ax.set_yticks([0.2,0.5,1,2,5,10,20,50,100,200]); ax.yaxis.set_major_formatter(ScalarFormatter())
        ax.set_title(f"$q={q}$")
        ax.set_xlabel(r"Group size $N$")
        ax.grid(alpha=0.12)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    axes_mq[0].set_ylabel(r"Fitted $\hat\beta$")
    axes_mq[0].text(0.05, 0.9, "Coordinated", transform=axes_mq[0].transAxes, fontsize=8, color="#3A6B38")
    axes_mq[0].text(0.4, 0.315, "Partially coord.", transform=axes_mq[0].transAxes, fontsize=8, color="#8A6D1D")
    axes_mq[0].text(0.05, 0.06, "Uncoordinated", transform=axes_mq[0].transAxes, fontsize=8, color="#39597A")
    handles, labels = axes_mq[0].get_legend_handles_labels()
    handles.append(Line2D([0], [0], marker="^", markerfacecolor="white", markeredgecolor="0.3",
                          linestyle="none", markersize=6.5, label="poorly determined estimate"))
    labels.append("poorly determined estimate")
    fig_mq.legend(handles, labels, loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 1.14))
    fig_mq.tight_layout(rect=[0,0,1,0.99])
    save("fig-results-multiq-phase")

In [ ]:
llama_models = ["llama31_8b_it", "llama3_70b_awq"]
drift_qs = [10, 50]
drift_display = 199

def upper_crossing(g):
    g = g.sort_values("leader_share").reset_index(drop=True)
    d = g["p_leader"] - g["leader_share"]
    crossing = None
    for i in range(len(g) - 1):
        if d[i] > 0 and d[i + 1] <= 0:
            x0, x1 = g["leader_share"][i], g["leader_share"][i + 1]
            y0, y1 = d[i], d[i + 1]
            cross = x0 + y0 * (x1 - x0) / (y0 - y1)
            crossing = (cross, i)
    return crossing

with plt.rc_context(THESIS_STYLE):
    fig_d, axes_d = plt.subplots(1, 2, figsize=(9.0, 3.8), sharex=True, sharey=True)

    for ax, model in zip(axes_d, llama_models):
        for q in drift_qs:
            g = response[
                (response["model_label"] == model)
                & (response["q"] == q)
                & (response["n_display"] == drift_display)
            ].sort_values("leader_share")
            if g.empty:
                continue
            crossing = upper_crossing(g)
            if crossing is not None:
                cross, before_cross = crossing
                measured_curve = g.iloc[:before_cross + 1]
                x_curve = list(measured_curve["leader_share"]) + [cross]
                y_curve = list(measured_curve["p_leader"]) + [cross]
            else:
                measured_curve = g
                x_curve = g["leader_share"]
                y_curve = g["p_leader"]

            ax.plot(
                x_curve, y_curve, linewidth=1.2, color=q_colors[q],
            )
            ax.scatter(
                measured_curve["leader_share"], measured_curve["p_leader"],
                marker="o", s=13, color=q_colors[q], zorder=4,
            )
            if crossing is not None:
                ax.scatter(
                    cross, cross, s=55, facecolors="none",
                    edgecolors=q_colors[q], linewidths=1.5, zorder=5,
                )
            finals = summary[
                (summary["model_label"] == model)
                & (summary["q"] == q)
                & (summary["N"] == drift_display + 1)
            ]["leader_share"]
            final_mean = finals.mean()
            ax.errorbar(
                final_mean, final_mean,
                xerr=[[final_mean - finals.min()], [finals.max() - final_mean]],
                fmt="D", markersize=5.5, color=q_colors[q],
                markeredgecolor="black", markeredgewidth=0.8,
                ecolor=q_colors[q], elinewidth=1.2, capsize=3, zorder=6,
            )

        ax.plot([0, 1], [0, 1], color="0.55", linestyle="--", linewidth=0.9)
        ax.set_title(model_names[model])
        ax.set_xlim(0, 1.02)
        ax.set_ylim(-0.02, 1.03)
        ax.set_xlabel(r"Leader share $s$")
        style_axis(ax)

    axes_d[0].set_ylabel(r"$P_{\mathrm{lead}}(s)$")
    drift_legend = [
        Line2D([0], [0], color=q_colors[q], marker="o", markersize=3.6,
               linewidth=1.2, label=f"$q={q}$")
        for q in drift_qs
    ]
    drift_legend.append(Line2D([0], [0], color="0.55", linestyle="--", label=r"$P=s$"))
    drift_legend.append(Line2D([0], [0], marker="o", markerfacecolor="none",
                               markeredgecolor="0.3", linestyle="none", label="predicted rest"))
    drift_legend.append(Line2D([0], [0], marker="D", color="0.3", markeredgecolor="black",
                               linestyle="none", markersize=5, label="mean observed final"))
    fig_d.legend(handles=drift_legend, loc="upper center", ncol=5,
                 frameon=False, bbox_to_anchor=(0.5, 1.10))
    fig_d.tight_layout(rect=[0, 0, 1, 0.99])
    save("fig-results-llama-drift")

In [ ]:
pred_model = "qwen25_7b_it"
pred_N = 100
pred_display = pred_N - 1
pred_q = 50
s_max_pred = (pred_N - pred_q) / (pred_N - 1)

# Refit the supplementary binary states
endgame_states = pd.read_csv(plot_data / "endgame_same_label_test.csv")
endgame_states = endgame_states[
    (endgame_states["model"] == pred_model)
    & (endgame_states["N"] == pred_N)
    & (endgame_states["displayed"] == pred_display)
    & (endgame_states["labels"] == "q50_pool")
].sort_values("n_lead")
binary_n = endgame_states["queries"].to_numpy()
binary_k = np.rint(binary_n * endgame_states["p_lead"]).astype(int)
binary_m = (
    (endgame_states["n_lead"] - endgame_states["n_other"])
    / endgame_states["displayed"]
).to_numpy()

def fit_endgame_beta(k):
    k = np.asarray(k)
    lower = np.zeros(k.shape[:-1])
    upper = np.full(k.shape[:-1], 100.0)
    for _ in range(80):
        beta = (lower + upper) / 2
        p = 0.5 * (1 + np.tanh(beta[..., None] * binary_m))
        score = np.sum(binary_m * (k - binary_n * p), axis=-1)
        lower = np.where(score > 0, beta, lower)
        upper = np.where(score > 0, upper, beta)
    return (lower + upper) / 2

binary_beta = float(fit_endgame_beta(binary_k))
# Empirical bootstrap within each measured state
binary_rng = np.random.default_rng(20260906)
binary_resamples = binary_rng.binomial(
    binary_n, binary_k / binary_n, size=(20000, len(binary_n)),
)
binary_ci = np.quantile(fit_endgame_beta(binary_resamples), [0.025, 0.975])

with plt.rc_context(THESIS_STYLE):
    fig_p, (ax_r, ax_t) = plt.subplots(1, 2, figsize=(9.0, 3.8))

    # left: fitted conformity
    pred_fits = fits[
        (fits["model_label"] == pred_model)
        & (fits["n_display"] == pred_display)
        & (fits["q"] == pred_q)
    ].set_index("q").copy()
    pred_fits.loc[2, ["beta_m1", "beta_ci_lo", "beta_ci_hi"]] = [
        binary_beta, binary_ci[0], binary_ci[1],
    ]
    pred_thresholds = {
        pred_q: (
            (pred_q - 1) / (pred_q - 2) * np.log(pred_q - 1),
            0.5 * np.log((pred_q - 1) * pred_N),
        ),
        2: (1.0, 0.5 * np.log(pred_N)),
    }
    for x, q in enumerate([pred_q, 2]):
        row = pred_fits.loc[q]
        ax_r.errorbar(
            x, row["beta_m1"],
            yerr=[[row["beta_m1"] - row["beta_ci_lo"]],
                  [row["beta_ci_hi"] - row["beta_m1"]]],
            fmt="o", ms=6, color=q_colors[q], capsize=3,
        )
        beta_c_q, beta_t_q = pred_thresholds[q]
        ax_r.hlines(
            beta_c_q, x - 0.22, x + 0.22, color="0.35",
            linestyle="--", linewidth=1.1,
            label=r"$\beta_c(q)$" if x == 0 else None,
        )
        ax_r.hlines(
            beta_t_q, x - 0.22, x + 0.22, color="0.35",
            linestyle=":", linewidth=1.4,
            label=r"$\beta_t(N,q)$" if x == 0 else None,
        )
    ax_r.set_yscale("log")
    ax_r.set_xlim(-0.5, 1.5)
    ax_r.set_ylim(0.7, 40)
    ax_r.set_yticks([1, 2, 5, 10, 20])
    ax_r.yaxis.set_major_formatter(ScalarFormatter())
    ax_r.set_xticks([0, 1])
    ax_r.set_xticklabels([r"$q=50$", "$q=2$\n(fifty-label pool)"])
    ax_r.set_ylabel(r"$\beta$")
    ax_r.set_title("Fitted conformity and benchmarks, $N=100$")
    ax_r.legend(frameon=False, loc="upper right", fontsize=8)
    style_axis(ax_r)

    # right: leader-share trajectories
    pred_runs = trajectory_data[
        (trajectory_data["model_label"] == pred_model)
        & (trajectory_data["q"] == pred_q)
        & (trajectory_data["N"] == pred_N)
    ]

    def split_at_k2(g):
        t_k2 = g.loc[g["n_active_opinions"] <= 2, "time"].min()
        multi = g[g["time"] <= t_k2]
        binary = g[g["time"] >= multi["time"].max()]
        return multi, binary

    for run, g in pred_runs.groupby("run"):
        multi, binary = split_at_k2(g.sort_values("time"))
        ax_t.plot(
            multi["time"], multi["leader_share"],
            color=q_colors[pred_q], linewidth=0.8, alpha=0.3,
        )
        ax_t.plot(
            binary["time"], binary["leader_share"],
            color=q_colors[2], linewidth=0.8, alpha=0.3,
        )

    mean_run = (
        pred_runs.groupby("time", as_index=False)
        .agg(
            leader_share=("leader_share", "mean"),
            n_active_opinions=("n_active_opinions", "mean"),
        )
        .sort_values("time")
    )
    multi, binary = split_at_k2(mean_run)
    ax_t.plot(
        multi["time"], multi["leader_share"], color=q_colors[pred_q],
        linewidth=1.8, label=r"$K(t)>2$",
    )
    ax_t.plot(
        binary["time"], binary["leader_share"], color=q_colors[2],
        linewidth=1.8, label=r"$K(t)=2$",
    )

    ax_t.set_xlim(0, 100)
    ax_t.set_ylim(0, 1.02)
    ax_t.set_xlabel(r"Time $t$")
    ax_t.set_ylabel(r"Leader share $s(t)$")
    ax_t.set_title("Opinion dynamics, $q=50$, $N=100$")
    ax_t.legend(frameon=False, loc="lower right", fontsize=8)
    style_axis(ax_t)

    fig_p.tight_layout()
    save("fig-results-qwen-twostage")

In [ ]:
beta_eff = pd.read_csv(plot_data / "effective_parameters.csv")
if legacy_beta_units:
    for column in ["beta_eff", "beta_eff_ci_lo", "beta_eff_ci_hi"]:
        beta_eff[column] = beta_eff[column] / 2

interior = beta_eff[~beta_eff["at_boundary"]].sort_values("lead")
smallest = interior.groupby(
    ["model_label", "q", "n_display"], as_index=False,
).first()
smallest = smallest[~smallest["q"].isin([3, 5, 25])]

with plt.rc_context(THESIS_STYLE):
    fig_bq, ax_q = plt.subplots(figsize=(6.0, 4.0))
    panel = smallest[smallest["n_display"] == 199]
    for model in models:
        group = panel[panel["model_label"] == model].sort_values("q")
        if group.empty:
            continue
        ax_q.plot(
            group["q"], group["beta_eff"], marker="o", linewidth=1.4,
            markersize=4, color=model_colors[model], label=model_names[model],
        )
    ax_q.axhline(0, color="0.55", linestyle=":", linewidth=0.8)
    q_ticks = sorted(panel["q"].unique())
    ax_q.set_xscale("log")
    ax_q.set_xticks(q_ticks)
    ax_q.set_xticklabels(q_ticks)
    ax_q.xaxis.set_minor_locator(plt.NullLocator())
    ax_q.set_xlabel(r"Number of opinions $q$")
    ax_q.set_ylabel(r"Conformity strength $\beta$")
    style_axis(ax_q)
    handles, labels = ax_q.get_legend_handles_labels()
    fig_bq.legend(
        handles, labels, loc="upper center", ncol=3,
        frameon=False, bbox_to_anchor=(0.5, 1.13), fontsize=8,
    )
    fig_bq.tight_layout(rect=[0, 0, 1, 0.93])
    save("fig-results-beta-effective")

In [ ]:
profile_model, profile_q, profile_display = "qwen25_32b_it", 50, 199
profile = interior[
    (interior["model_label"] == profile_model)
    & (interior["q"] == profile_q)
    & (interior["n_display"] == profile_display)
].sort_values("leader_share")
profile_fit = fits[
    (fits["model_label"] == profile_model)
    & (fits["q"] == profile_q)
    & (fits["n_display"] == profile_display)
]["beta_m1"].iloc[0]

with plt.rc_context(THESIS_STYLE):
    fig_bp, ax_p = plt.subplots(figsize=(5.6, 3.8))
    color = model_colors[profile_model]
    ax_p.errorbar(
        profile["leader_share"], profile["beta_eff"],
        yerr=[profile["beta_eff"] - profile["beta_eff_ci_lo"],
              profile["beta_eff_ci_hi"] - profile["beta_eff"]],
        fmt="o-", markersize=4, linewidth=1.3, color=color, capsize=2.5,
        label=r"$\beta_{\mathrm{eff}}(s)$",
    )
    ax_p.axhline(profile_fit, color="0.35", linestyle="-", linewidth=1.0,
                 label=r"fitted $\hat\beta$")
    ax_p.set_xlim(0, 0.8)
    ax_p.set_ylim(0, 58)
    ax_p.set_xlabel(r"Leader share $s$")
    ax_p.set_ylabel(r"$\beta_{\mathrm{eff}}(s)$")
    ax_p.legend(frameon=False, loc="upper right", fontsize=8)
    style_axis(ax_p)
    fig_bp.tight_layout()
    save("fig-results-beta-eff-profile")

In [ ]:
binary_profile_model = "qwen25_32b_it"

with plt.rc_context(THESIS_STYLE):
    fig_b2, ax = plt.subplots(figsize=(5.6, 3.8))
    color = model_colors[binary_profile_model]
    g = interior[
        (interior["model_label"] == binary_profile_model)
        & (interior["q"] == 2)
        & (interior["n_display"] == 199)
    ].sort_values("leader_share")
    b_fit = fits[
        (fits["model_label"] == binary_profile_model)
        & (fits["q"] == 2)
        & (fits["n_display"] == 199)
    ]["beta_m1"].iloc[0]
    ax.errorbar(
        g["leader_share"], g["beta_eff"],
        yerr=[g["beta_eff"] - g["beta_eff_ci_lo"],
              g["beta_eff_ci_hi"] - g["beta_eff"]],
        fmt="o-", markersize=4, linewidth=1.3, color=color, capsize=2.5,
        label=r"$\beta_{\mathrm{eff}}(s)$",
    )
    ax.axhline(b_fit, color="0.35", linestyle="-", linewidth=1.0,
               label=r"fitted $\hat\beta$")
    ax.set_xlim(0.5, 1.0)
    ax.set_ylim(0, 8)
    ax.set_xlabel(r"Leader share $s$")
    ax.set_ylabel(r"$\beta_{\mathrm{eff}}(s)$")
    ax.set_title(f"{model_names[binary_profile_model]}, $q=2$, $N=200$", fontsize=9)
    ax.legend(frameon=False, fontsize=8)
    style_axis(ax)
    fig_b2.tight_layout()
    save("fig-results-beta-eff-binary")

## Appendix A

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.lines import Line2D

THESIS_STYLE = {
    "font.family": "serif",
    "font.serif": ["STIX Two Text", "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 9,
    "axes.linewidth": 0.7,
}

model_names = {
    "qwen25_7b_it": "Qwen 2.5 7B",
    "qwen25_32b_it": "Qwen 2.5 32B",
    "gemma4_E4B_it": "Gemma 4 E4B",
    "gemma4_31B_dense": "Gemma 4 31B Dense",
}

model_colors = {
    "qwen25_7b_it": "#FF7F0E",
    "qwen25_32b_it": "#D62728",
    "gemma4_E4B_it": "#17BECF",
    "gemma4_31B_dense": "#1F77B4",
}

def style_axis(ax):
    ax.grid(alpha=0.18)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

struct_val = pd.read_csv(Path("../data/plot_sources/structured_validation.csv"))
struct_models = ["gemma4_E4B_it", "gemma4_31B_dense", "qwen25_7b_it", "qwen25_32b_it"]
state_markers = {
    "online_control": ("o", "even-rest control"),
    "one_agent_lead": ("v", "one-agent lead"),
    "staircase": ("s", "graded support"),
    "strong_runner": ("^", "strong runner-up"),
    "two_leader_tie": ("D", "two-leader tie"),
}

with plt.rc_context(THESIS_STYLE):
    fig_s, ax_s = plt.subplots(figsize=(6.0, 4.4))
    ax_s.plot([0, 1], [0, 1], color="0.55", linestyle="--", linewidth=0.9)
    for r in struct_val.itertuples():
        if r.model_label not in struct_models:
            continue
        marker = state_markers[r.state.split("_", 1)[1]][0]
        ax_s.scatter(
            r.M1_P_leader, r.obs_P_leader, marker=marker, s=34,
            color=model_colors[r.model_label], edgecolors="black",
            linewidths=0.4, zorder=4,
        )
    ax_s.set_xlim(-0.02, 1.02)
    ax_s.set_ylim(-0.02, 1.02)
    ax_s.set_xlabel("Predicted leader-adoption probability")
    ax_s.set_ylabel("Observed leader-adoption frequency")
    style_axis(ax_s)
    state_legend = [
        Line2D([0], [0], marker=m, color="0.3", linestyle="none", markersize=5, label=lab)
        for m, lab in state_markers.values()
    ]
    model_legend = [
        Line2D([0], [0], marker="o", color=model_colors[m], linestyle="none",
               markersize=5, label=model_names[m])
        for m in struct_models
    ]
    first_legend = ax_s.legend(
        handles=model_legend, loc="upper left", frameon=False, fontsize=7.5,
    )
    ax_s.add_artist(first_legend)
    ax_s.legend(
        handles=state_legend, loc="lower right", frameon=False, fontsize=7.5,
    )
    fig_s.tight_layout()
    save("fig-structured-state-validation")

## Appendix B

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

sem_dir = Path("../data/plot_sources")

sem_options = {
    "energy_q2_anchor": ["renewable energy", "fossil fuels"],
    "speech_q3": [
        "unrestricted free speech", "regulated speech",
        "harm-reduction moderation",
    ],
    "ai_governance_q4": [
        "open-source AI", "corporate self-regulation",
        "government regulation", "international AI treaty",
    ],
    "political_ideology_q6": [
        "liberalism", "conservatism", "socialism", "libertarianism",
        "green politics", "nationalism",
    ],
    "policy_priorities_q10": [
        "healthcare", "education", "climate policy", "economic growth",
        "public safety", "housing", "immigration", "digital rights",
        "national defense", "social welfare",
    ],
}
sem_fits = pd.read_csv(sem_dir / "semantic_parameters.csv")
sem_fits = sem_fits[sem_fits["set_name"].isin(sem_options)].copy()
sem_h = pd.read_csv(sem_dir / "semantic_fields.csv")
sem_h = sem_h[sem_h["set_name"].isin(sem_options)].copy()
# the public table has one row per (state, reply) with a count n;
# expand it back to one row per query so the code below stays as it was
sem_counts = pd.read_csv(sem_dir / "semantic_response_counts.csv")
sem_queries = sem_counts.loc[sem_counts.index.repeat(sem_counts["n"])]
sem_queries = sem_queries.drop(columns="n").reset_index(drop=True)
sem_queries = sem_queries[sem_queries["set_name"].isin(sem_options)].copy()
sem_q = sem_queries[
    sem_queries["valid"] & sem_queries["chosen_option_idx"].notna()
].copy()
sem_q["shares"] = sem_q["shares_options"].apply(json.loads)
sem_q["chosen"] = sem_q["chosen_option_idx"].astype(int)

SEM_STYLE = {
    "font.family": "serif",
    "font.serif": ["STIX Two Text", "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "axes.linewidth": 0.7,
}
sem_models = [
    "llama31_8b_it", "llama3_70b_awq", "qwen25_7b_it",
    "qwen25_32b_it", "gemma4_E4B_it", "gemma4_31B_dense",
]
sem_model_names = {
    "llama31_8b_it": "Llama 3.1\n8B",
    "llama3_70b_awq": "Llama 3\n70B",
    "qwen25_7b_it": "Qwen 2.5\n7B",
    "qwen25_32b_it": "Qwen 2.5\n32B",
    "gemma4_E4B_it": "Gemma 4\nE4B",
    "gemma4_31B_dense": "Gemma 4\n31B Dense",
}

def sem_field_vector(model, set_name):
    fields = sem_h[
        (sem_h["model_label"] == model) & (sem_h["set_name"] == set_name)
    ].set_index("option")["h"].reindex(sem_options[set_name])
    if fields.isna().any():
        raise ValueError(f"Missing semantic fields: {model}, {set_name}")
    return fields.to_numpy()

def sem_leader_curves(set_name, option_colors, x_min, x_ticks, legend_cols):
    """Observed adoption frequencies for each option in the designated leader role."""
    options = sem_options[set_name]
    with plt.rc_context(SEM_STYLE):
        fig, axes = plt.subplots(2, 3, figsize=(6.4, 4.3), sharex=True, sharey=True)
        for ax, model in zip(axes.flat, sem_models):
            group = sem_q[
                (sem_q["model_label"] == model)
                & (sem_q["set_name"] == set_name)
            ]
            for idx, color in enumerate(option_colors):
                lead = group[group["leader_option_idx"] == idx].copy()
                lead["s"] = lead["shares"].apply(lambda shares: shares[idx])
                lead["chosen_leader"] = lead["chosen"] == idx
                observed = lead.groupby("s")["chosen_leader"].mean().sort_index()
                ax.plot(
                    observed.index, observed.values, "o-", color=color,
                    markersize=2.8, linewidth=1.0,
                )
            ax.set_title(sem_model_names[model].replace("\n", " "), fontsize=9)
            ax.set_xlim(x_min, 1.0)
            ax.set_ylim(-0.03, 1.03)
            ax.set_xticks(x_ticks)
            ax.set_yticks([0, 0.25, 0.5, 0.75, 1])
            ax.tick_params(labelbottom=True, labelleft=True)
            ax.grid(alpha=0.15)
            ax.spines[["top", "right"]].set_visible(False)
        handles = [
            Line2D([], [], color=color, marker="o", markersize=2.8,
                   linewidth=1.0, label=option)
            for option, color in zip(options, option_colors)
        ]
        fig.legend(
            handles=handles, loc="upper center", ncol=legend_cols,
            frameon=False, fontsize=7.5, bbox_to_anchor=(0.5, 0.995),
            borderaxespad=0, columnspacing=1.8, handlelength=2.0,
        )
        fig.supxlabel(r"Displayed share of the option $s$", y=0.015, fontsize=9)
        fig.supylabel(r"Adoption probability $P(\alpha)$", x=0.005, fontsize=9)
        fig.subplots_adjust(
            left=0.095, right=0.99, bottom=0.12, top=0.81,
            wspace=0.27, hspace=0.48,
        )
        pass

In [ ]:
with plt.rc_context(SEM_STYLE):
    fig, axes = plt.subplots(1, 2, figsize=(6.2, 3.3), sharex=True, sharey=True)
    for ax, model, color in zip(
        axes,
        ["qwen25_7b_it", "gemma4_31B_dense"],
        ["#FF7F0E", "#1F77B4"],
    ):
        g = sem_q[
            (sem_q["model_label"] == model)
            & (sem_q["set_name"] == "energy_q2_anchor")
        ].copy()
        g["m"] = g["shares"].apply(lambda x: x[0] - x[1])
        g["renewable"] = g["chosen"] == 0
        observed = g.groupby("m")["renewable"].mean().sort_index()
        fit = sem_fits[
            (sem_fits["model_label"] == model)
            & (sem_fits["set_name"] == "energy_q2_anchor")
        ].iloc[0]
        fields = sem_field_vector(model, "energy_q2_anchor")
        m_grid = np.linspace(observed.index.min(), observed.index.max(), 300)
        p_semantic = 0.5 * (
            1 + np.tanh(fit["beta"] * m_grid + (fields[0] - fields[1]) / 2)
        )
        p_neutral = 0.5 * (1 + np.tanh(fit["neutral_beta_gridmatched"] * m_grid))
        ax.plot(m_grid, p_neutral, color="0.45", linestyle="--", linewidth=1.2)
        ax.plot(m_grid, p_semantic, color=color, linewidth=1.5)
        ax.plot(observed.index, observed.values, "o", color=color, markersize=3.7)
        ax.axvline(0, color="0.75", linewidth=0.7, linestyle=":")
        ax.set_title(sem_model_names[model].replace("\n", " "))
        ax.set_xlim(-1.03, 1.03)
        ax.set_ylim(-0.03, 1.03)
        ax.set_xticks([-1, -0.5, 0, 0.5, 1])
        ax.set_yticks([0, 0.25, 0.5, 0.75, 1])
        ax.tick_params(labelbottom=True, labelleft=True)
        ax.set_xlabel(r"Collective opinion $m$")
        ax.set_ylabel(r"Adoption probability $P(m)$")
        ax.grid(axis="y", alpha=0.18)
        ax.spines[["top", "right"]].set_visible(False)
    fig.legend(
        handles=[
            Line2D([], [], color="0.25", marker="o", linestyle="none", markersize=3.7,
                   label="Measured adoption frequency"),
            Line2D([], [], color="0.25", label="Biased fit"),
            Line2D([], [], color="0.45", linestyle="--", label="Neutral-label fit"),
        ],
        loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.01),
    )
    fig.tight_layout(rect=[0, 0, 1, 0.9], w_pad=1.3)
    save("fig-semantic-anchor")

In [ ]:
sem_leader_curves(
    "political_ideology_q6",
    ["#1F77B4", "#D62728", "#9467BD", "#FF7F0E", "#2CA02C", "#8C564B"],
    x_min=0.12, x_ticks=[0.2, 0.4, 0.6, 0.8, 1.0], legend_cols=3,
)
save("fig-semantic-ideology")

In [ ]:
sem_leader_curves(
    "ai_governance_q4",
    ["#1F77B4", "#D62728", "#2CA02C", "#FF7F0E"],
    x_min=0.20, x_ticks=[0.25, 0.5, 0.75, 1.0], legend_cols=2,
)
save("fig-semantic-governance")

## Appendix C

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import NullLocator, ScalarFormatter

appc_dir = Path("../data/plot_sources")

APPC_PARENT = "qwen25_7b_parent"
APPC_ABL = "qwen25_7b_abliterated_v2"
APPC_MODELS = [APPC_PARENT, APPC_ABL]
APPC_COLORS = {APPC_PARENT: "#FF7F0E", APPC_ABL: "#6A3D9A"}
APPC_NAMES = {
    APPC_PARENT: "Unaltered model",
    APPC_ABL: "Abliterated model",
}
APPC_SEMANTIC = "political_ideology_q6_N50"
APPC_MATCHED = "neutral_matched_political_q6_N50"
APPC_NEUTRAL = [
    "neutral_q2_N100", "neutral_q10_N100",
    "neutral_q10_N200", "neutral_q50_N200",
]
APPC_IDEOLOGIES = [
    "liberalism", "conservatism", "socialism", "libertarianism",
    "green politics", "nationalism",
]
APPC_IDEOLOGY_COLORS = [
    "#1F77B4", "#D62728", "#9467BD", "#FF7F0E", "#2CA02C", "#8C564B",
]
APPC_STYLE = {
    "font.family": "serif",
    "font.serif": ["STIX Two Text", "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "axes.linewidth": 0.7,
    "pdf.fonttype": 42,
}

appc_summary = pd.read_csv(appc_dir / "abliterated_dynamics_summary.csv")
appc_trajectories = pd.read_csv(appc_dir / "abliterated_dynamics_trajectories.csv")
# the public table has one row per (state, reply) with a count n;
# expand it back to one row per query so the code below stays as it was
appc_counts = pd.read_csv(appc_dir / "abliterated_response_counts.csv")
appc_queries = appc_counts.loc[appc_counts.index.repeat(appc_counts["n"])]
appc_queries = appc_queries.drop(columns="n").reset_index(drop=True)

appc_regular = appc_trajectories[
    appc_trajectories["record_type"].eq("regular")
].copy()
appc_keys = ["model_label", "condition_id", "run"]
appc_regular["C"] = (
    appc_regular["q"] * appc_regular["leader_share"] - 1
) / (appc_regular["q"] - 1)

# Count valid replies with the ideology
# chosen_role == 0 is leader adoption
appc_valid = appc_queries["valid"].astype(str).str.lower().isin(["true", "1"])
appc_semantic_queries = appc_queries[
    appc_valid & appc_queries["block_id"].eq(APPC_SEMANTIC)
].copy()
appc_semantic_queries["leader_chosen"] = appc_semantic_queries["chosen_role"].eq(0)
appc_semantic = appc_semantic_queries.groupby(
    ["model_label", "designated_leader_label", "leader_share"], as_index=False
).agg(
    n=("leader_chosen", "size"),
    k=("leader_chosen", "sum"),
    p=("leader_chosen", "mean"),
)

def appc_style_axis(ax):
    ax.grid(alpha=.18)
    ax.spines[["top", "right"]].set_visible(False)


def appc_time_axis(ax, end=100):
    ticks = [0, 1, 2, 5, 10, 20, 50, 100] if end == 100 else [0, 1, 2, 5, 10]
    ax.set_xscale("function", functions=(np.log1p, np.expm1))
    ax.set_xlim(0, end)
    ax.set_xticks(ticks)
    ax.set_xticklabels(ticks)
    appc_style_axis(ax)


def appc_active_axis(ax, q):
    ax.set_yscale("log")
    ticks = [k for k in [1, 2, 5, 10, 20, 50] if k <= q]
    ax.set_yticks(ticks)
    ax.yaxis.set_major_formatter(ScalarFormatter())
    ax.yaxis.set_minor_locator(NullLocator())
    ax.set_ylim(.91, q * 1.13)


def appc_plot_dynamics(ax, model, block, variable, linestyle="-", individuals=False):
    g = appc_regular[(appc_regular.model_label == model) & (appc_regular.condition_id == block)]
    q = int(g.q.iloc[0])
    if individuals:
        for _, run in g.groupby("run"):
            run = run.sort_values("time_sweeps")
            reached = run[run.n_active_opinions.eq(1)]
            if len(reached):
                run = run[run.time_sweeps <= reached.time_sweeps.iloc[0]]
            ax.plot(run.time_sweeps, run[variable], color=APPC_COLORS[model],
                    ls=linestyle, lw=.55, alpha=.22, zorder=1,
                    drawstyle="steps-post" if variable == "n_active_opinions" else "default")
    mean = g.groupby("time_sweeps")[variable].mean().sort_index()
    done = g.groupby("time_sweeps").n_active_opinions.mean().eq(1)
    if done.any():
        first = done[done].index[0]
        mean = mean.loc[:first]
    ax.plot(mean.index, mean.values, color=APPC_COLORS[model], ls=linestyle, lw=1.5, zorder=3,
            drawstyle="steps-post" if variable == "n_active_opinions" else "default")
    if done.any():
        ax.scatter(first, mean.iloc[-1], color=APPC_COLORS[model], marker="v", s=17,
                   facecolors=APPC_COLORS[model] if linestyle == "-" else "white", linewidths=.8, zorder=4)
    return q

In [ ]:
with plt.rc_context(APPC_STYLE):
    fig, axes = plt.subplots(2, 4, figsize=(7.2, 3.8), sharex=True)
    for col, block in enumerate(APPC_NEUTRAL[:4]):
        for model in APPC_MODELS:
            q = appc_plot_dynamics(axes[0, col], model, block, "C", individuals=True)
            appc_plot_dynamics(axes[1, col], model, block, "n_active_opinions", individuals=True)
        N = int(appc_summary[appc_summary.condition_id.eq(block)].N.iloc[0])
        axes[0, col].set_title(f"$q={q},\\ N={N}$", fontsize=9)
        axes[0, col].set_ylim(-.03, 1.06)
        axes[0, col].set_yticks([0, .5, 1])
        axes[0, col].axhline(1, ls="--", color=".5", lw=.65)
        appc_active_axis(axes[1, col], q)
        for row in range(2):
            appc_time_axis(axes[row, col])
            axes[row, col].set_xticks([0, 1, 5, 20, 100])
            axes[row, col].set_xticklabels([0, 1, 5, 20, 100])
        axes[1, col].set_xlabel(r"Time $t$ (sweeps)", fontsize=8)
    axes[0, 0].set_ylabel(r"Coordination $C(t)$")
    axes[1, 0].set_ylabel(r"Active opinions $K(t)$")
    handles = [Line2D([], [], color=APPC_COLORS[m], lw=1.5, label=APPC_NAMES[m]) for m in APPC_MODELS]
    fig.legend(handles=handles, loc="upper center", ncol=2, frameon=False,
               bbox_to_anchor=(.52, 1.01))
    fig.subplots_adjust(left=.075, right=.988, bottom=.135, top=.82, wspace=.28, hspace=.23)
    save("fig-appc-neutral")

In [ ]:
with plt.rc_context(APPC_STYLE):
    fig, axes = plt.subplots(2, 2, figsize=(7.4, 5.65))
    for model in APPC_MODELS:
        for block, ls in [(APPC_MATCHED, "--"), (APPC_SEMANTIC, "-")]:
            appc_plot_dynamics(axes[0, 0], model, block, "C", ls)
            appc_plot_dynamics(axes[0, 1], model, block, "n_active_opinions", ls)
    for ax in axes[0]:
        appc_time_axis(ax, 12)
        ax.set_xlabel(r"Time $t$ (sweeps)")
    axes[0, 0].set_ylabel(r"Coordination $C(t)$")
    axes[0, 0].set_ylim(-.03, 1.06)
    axes[0, 0].set_yticks([0, .25, .5, .75, 1])
    axes[0, 0].set_title(r"(a) Coordination, $q=6,\ N=50$", fontsize=9)
    axes[0, 1].set_title(r"(b) Active opinions, $q=6,\ N=50$", fontsize=9)
    appc_active_axis(axes[0, 1], 6)
    axes[0, 1].set_yticks([1, 2, 3, 6])
    axes[0, 1].set_ylabel(r"Active opinions $K(t)$")
    handles = [Line2D([], [], color=APPC_COLORS[m], ls=ls, lw=1.5,
                     label=f"{APPC_NAMES[m]}, {lab}")
               for m in APPC_MODELS for lab, ls in [("neutral", "--"), ("ideology", "-")]]
    fig.legend(handles=handles, loc="upper center", ncol=2, frameon=False,
               bbox_to_anchor=(.52, 1.01), fontsize=8)
    for ax, model, panel in zip(axes[1], APPC_MODELS, ["c", "d"]):
        for option, color in zip(APPC_IDEOLOGIES, APPC_IDEOLOGY_COLORS):
            curve = appc_semantic[(appc_semantic.model_label == model) &
                             (appc_semantic.designated_leader_label == option)].sort_values("leader_share")
            ax.plot(curve.leader_share, curve.p, "o-", color=color, ms=2.8, lw=1)
        ax.set_xlim(.12, 1)
        ax.set_ylim(-.03, 1.03)
        ax.set_xticks([.2, .4, .6, .8, 1])
        ax.set_yticks([0, .25, .5, .75, 1])
        ax.set_xlabel(r"Displayed share of the option $s$")
        ax.set_ylabel(r"Adoption probability $P(\alpha)$")
        ax.set_title(f"({panel}) {APPC_NAMES[model]}", fontsize=9)
        appc_style_axis(ax)
    handles = [Line2D([], [], color=c, marker="o", ms=2.8, lw=1, label=o)
               for o, c in zip(APPC_IDEOLOGIES, APPC_IDEOLOGY_COLORS)]
    fig.legend(handles=handles, loc="upper center", ncol=3, frameon=False,
               bbox_to_anchor=(.52, .485), fontsize=8, columnspacing=1.7)
    fig.subplots_adjust(left=.088, right=.99, bottom=.085, top=.84, wspace=.30, hspace=.85)
    save("fig-appc-semantic")